In [0]:
import re
import uuid
from datetime import datetime, timezone

from pyspark.sql import functions as F


AUDIT_TABLE = "tx_dev.control.tax_ingestion_audit"


def standardize_column_name(column_name):
    """Convert a column name to lowercase snake_case."""
    column_name = column_name.strip()
    column_name = re.sub(r"[^a-zA-Z0-9]+", "_", column_name)
    return column_name.strip("_").lower()


def ingest_csv(
    source_path,
    target_table,
    required_columns,
    source_system="TAX_REPORTING",
    load_type="FULL",
    dry_run=False,
):
    """Ingest a CSV file into a Bronze Delta table with audit logging."""

    run_id = str(uuid.uuid4())
    start_time = datetime.now(timezone.utc)

    source_file_name = source_path.rsplit("/", 1)[-1]
    source_record_count = 0
    target_record_count = 0
    status = "STARTED"
    error_message = None

    try:
        # Read source
        df = (
            spark.read
            .option("header", True)
            .option("inferSchema", True)
            .csv(source_path)
        )

        # Standardize columns
        for old_name in df.columns:
            df = df.withColumnRenamed(
                old_name,
                standardize_column_name(old_name),
            )

        source_record_count = df.count()

        # Validate required columns
        missing_columns = [
            column
            for column in required_columns
            if column not in df.columns
        ]

        if missing_columns:
            raise ValueError(
                f"Required columns are missing: {missing_columns}"
            )

        # Validate required values
        null_condition = F.lit(False)

        for column in required_columns:
            null_condition = (
                null_condition | F.col(column).isNull()
            )

        invalid_count = df.filter(null_condition).count()

        if invalid_count > 0:
            raise ValueError(
                f"{invalid_count} records contain null required values."
            )

        # Add ingestion metadata
        df = (
            df
            .withColumn(
                "source_file_name",
                F.lit(source_file_name),
            )
            .withColumn(
                "source_system",
                F.lit(source_system),
            )
            .withColumn(
                "ingestion_timestamp",
                F.current_timestamp(),
            )
            .withColumn(
                "ingestion_date",
                F.current_date(),
            )
        )

        # Write to target
        if dry_run:
            status = "DRY_RUN_SUCCESS"
        else:
            (
                df.write
                .format("delta")
                .mode("overwrite" if load_type == "FULL" else "append")
                .saveAsTable(target_table)
            )

            target_record_count = spark.table(target_table).count()
            status = "SUCCESS"

    except Exception as exc:
        status = "FAILED"
        error_message = str(exc)[:4000]

    # Write audit record
    end_time = datetime.now(timezone.utc)

    audit_schema = """
        run_id STRING,
        source_system STRING,
        source_file_name STRING,
        target_table STRING,
        load_type STRING,
        start_time TIMESTAMP,
        end_time TIMESTAMP,
        source_record_count BIGINT,
        target_record_count BIGINT,
        status STRING,
        error_message STRING
    """

    audit_record = [(
        run_id,
        source_system,
        source_file_name,
        target_table,
        load_type,
        start_time,
        end_time,
        source_record_count,
        target_record_count,
        status,
        error_message,
    )]

    audit_df = spark.createDataFrame(
        audit_record,
        schema=audit_schema,
    )

    (
        audit_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(AUDIT_TABLE)
    )

    print(f"Run ID: {run_id}")
    print(f"Target: {target_table}")
    print(f"Source records: {source_record_count}")
    print(f"Target records: {target_record_count}")
    print(f"Status: {status}")

    if status == "FAILED":
        raise RuntimeError(
            f"Ingestion failed for {source_file_name}: {error_message}"
        )

    return {
        "run_id": run_id,
        "status": status,
        "source_record_count": source_record_count,
        "target_record_count": target_record_count,
    }